# 01 · Data, features and baseline
Football injury risk prediction on SoccerMon (Zenodo record 10033832).

Run top to bottom in Colab. Section 2 downloads the data; section 3 is where you map the real column names once you've seen them.

## 1. Setup

In [ ]:
!git clone https://github.com/<your-username>/football-injury-risk.git 2>/dev/null || true
%cd football-injury-risk
!pip install -q -r requirements.txt

In [ ]:
import sys, os, json, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "src")
from features import complete_calendar, add_load_features, FEATURES
os.makedirs("data", exist_ok=True)

## 2. Download the dataset
The Zenodo API lists the files in the record. Check sizes before downloading, since the GPS part is huge and you don't need it for this project.

In [ ]:
rec = requests.get("https://zenodo.org/api/records/10033832", timeout=60).json()
for f in rec["files"]:
    print(f"{f['key']:<60} {f['size']/1e6:10.1f} MB")

In [ ]:
# Download only the small subjective files (training load, wellness, injury).
# Set WANTED after reading the listing above.
WANTED = []   # e.g. ["subjective.zip"]
for f in rec["files"]:
    if f["key"] in WANTED:
        url = f["links"]["self"]
        out = os.path.join("data", f["key"])
        with requests.get(url, stream=True, timeout=120) as r, open(out, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
        print("saved", out)

In [ ]:
# Unzip if needed, then look at what is there.
!cd data && for z in *.zip; do [ -f "$z" ] && unzip -oq "$z"; done; find . -maxdepth 3 -type f | head -50

## 3. Map columns to the schema the pipeline expects
Inspect the training-load and injury files, then fill in the mapping. The pipeline needs one row per player per day:

`player_id, date, load, injury`

- `load`: session RPE × duration (sum if there are several sessions in a day)
- `injury`: 1 on the day an injury was reported, else 0

In [ ]:
# Example only. Replace paths and column names after inspecting the files.
# load_raw = pd.read_csv("data/<training load file>.csv")
# inj_raw  = pd.read_csv("data/<injury file>.csv")
# load_raw.head(), inj_raw.head()

In [ ]:
# daily = ...  # build the 4-column frame here
# daily["date"] = pd.to_datetime(daily["date"])
# daily.to_csv("data/daily_load_injury.csv", index=False)

## 4. Features and target
Target = any injury onset in the next 7 days. Rolling features use past days only.

In [ ]:
# daily = pd.read_csv("data/daily_load_injury.csv", parse_dates=["date"])
# df = add_load_features(complete_calendar(daily), horizon=7)
# df = df.dropna(subset=["target"] + ["acute_7d", "chronic_28d"])
# print(df.shape, "positive rate:", df["target"].mean().round(4))

## 5. Split
Split by time, not by random row. Rows from neighbouring days are nearly identical, so a random split leaks information and inflates every metric. Train on the first season, validate on the early part of the second, test on the rest.

In [ ]:
# cut1, cut2 = pd.Timestamp("2021-01-01"), pd.Timestamp("2021-06-01")   # adjust to the real date range
# train = df[df.date < cut1]; val = df[(df.date >= cut1) & (df.date < cut2)]; test = df[df.date >= cut2]
# X = lambda d: d[FEATURES]; y = lambda d: d["target"].astype(int)

## 6. Baselines
Order matters: majority-class, then ACWR-only rule, then logistic regression, then LightGBM. Each has to beat the one before it.

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import make_pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
#
# lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
# lr.fit(X(train), y(train))
# p = lr.predict_proba(X(val))[:, 1]
# print("ROC-AUC", roc_auc_score(y(val), p), "PR-AUC", average_precision_score(y(val), p), "Brier", brier_score_loss(y(val), p))

## Next
- LightGBM + SHAP in `02_model.ipynb`
- Calibration plot and alert threshold (precision at a fixed number of alerts per team per week)
- Per-player error analysis